# Series Temporais: Fundamentos## Analise de Dados Sequenciais no Tempo**Objetivo**: Compreender conceitos basicos de series temporais, componentes, transformacoes e deteccao de padroes.**Contexto**: Series temporais aparecem em economia, meteorologia, saude, finanzas e IoT. Sao dados onde o tempo eh uma dimensao critica.

## 1. O que sao Series Temporais?Uma **serie temporal** eh uma sequencia de observacoes coletadas em instantes regulares (ou irregulares) de tempo.### Analogias Intuitivas:- **Batimento cardiaco**: pulsos registrados ao longo do tempo. Padrao regular com variacoes.- **Preco de acoes**: cotacoes que fluem, com tendencias e picos.- **Temperatura diaria**: ciclos sazonais (invierno mais frio, verao mais quente).- **Vendas mensais**: crescimento (tendencia) + picos em datas especiais (sazonalidade) + aleatoriedade (ruido).### O que observar:1. A natureza dependente do tempo dos dados2. A importancia da ordem cronologica3. Padroes que repetem (sazonalidade)4. Mudancas graduais (tendencias)5. Flutuacoes aleatorias sobrepostas### O que concluir:1. Series temporais nao sao dados independentes2. Metodos ML convencionais podem falhar3. Tempo eh dimensao fundamental4. Dependencia serial invalida hipoteses i.i.d.5. Ferramentas especializadas sao necessarias### Diferenca Fundamental:Em dados convencionais (ML padrao), a **ordem das linhas nao importa**. Em series temporais, a **ordem eh tudo**. O valor em t depende frequentemente de t-1, t-2, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Serie temporal simples: 100 dias
np.random.seed(42)
dias = np.arange(100)

# Preco de acao simulado: tendencia + sazonalidade + ruido
tendencia = dias * 0.5 + 50
sazonalidade = 5 * np.sin(2 * np.pi * dias / 30)
ruido = np.random.normal(0, 2, 100)
preco = tendencia + sazonalidade + ruido

plt.figure(figsize=(12, 4))
plt.plot(dias, preco, linewidth=2)
plt.xlabel("Dia")
plt.ylabel("Preco")
plt.title("Serie Temporal: Preco de Acao (100 dias)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/serie_temporal_simples.png", dpi=100, bbox_inches="tight")
plt.show()

print("Observacoes:")
print(f"Preco inicial: {preco[0]:.2f}")
print(f"Preco final: {preco[-1]:.2f}")
print(f"Minimo: {np.min(preco):.2f}")
print(f"Maximo: {np.max(preco):.2f}")


### O que observar:1. A serie tem uma tendencia de crescimento2. Flutuacoes regulares (sazonalidade) aparecem a cada ~30 dias3. Ruido aleatorio (variabilidade nao explicada) esta presente4. O valor em cada dia eh dependente dos dias anteriores### O que concluir:1. Series temporais sao compostas por multiplos componentes2. Visualizacao eh crucial para entender a estrutura3. Dependencia temporal invalida muitos metodos ML clasicos4. Precisamos de ferramentas especializadas para series temporais

## 2. Componentes de Series TemporaisToda serie temporal pode ser decomposta em:1. **Tendencia (Trend)**: direcao geral da serie a longo prazo (crescimento/queda)2. **Sazonalidade (Seasonality)**: padroes repetitivos em periodos fixos (diario, mensal, anual)3. **Ciclo**: flutuacoes longas (> 1 ano) nao periodicas estritamente4. **Ruido (Noise)**: variabilidade aleatoria irredutivel### Modelos de Decomposicao:- **Aditivo**: Y = Tendencia + Sazonalidade + Ruido- **Multiplicativo**: Y = Tendencia × Sazonalidade × RuidoEscolha aditivo quando componentes sao independentes (amplitudes fixas).Escolha multiplicativo quando sazonalidade cresce com tendencia.

In [ ]:
# Decomposicao aditiva manualmente
def decompose_aditiva(y, periodo):
    n = len(y)
    # Tendencia: media movel centrada
    tendencia = np.zeros(n)
    janela = periodo
    for i in range(n):
        inicio = max(0, i - janela // 2)
        fim = min(n, i + janela // 2 + 1)
        tendencia[i] = np.mean(y[inicio:fim])
    # Sazonalidade: media por posicao no ciclo
    y_detrended = y - tendencia
    sazonalidade = np.zeros(n)
    for fase in range(periodo):
        indices = np.arange(fase, n, periodo)
        if len(indices) > 0:
            sazonalidade[indices] = np.mean(y_detrended[indices])
    # Ruido: residuo
    ruido = y - tendencia - sazonalidade
    return tendencia, sazonalidade, ruido


### O que observar:1. Cada componente tem uma "escala" diferente2. Tendencia eh lisa (smooth), sazonalidade eh periodica, ruido eh aleatorio3. Somando os tres componentes recuperamos a serie original4. O ruido tem media aproximadamente zero### O que concluir:1. Decomposicao eh valiosa para entender a estrutura da serie2. Cada componente pode ser analisado separadamente3. Reducao de ruido (filtragem) eh importante antes de prever tendencias4. Periodicidade pode ser estimada visualmente

## 3. EstacionariedadeUma serie eh **estacionaria** se suas propriedades estatisticas (media, variancia, autocorrelacao) nao mudam ao longo do tempo.### Propriedades:- **Media constante**: E[Y_t] = mu (independente de t)- **Variancia constante**: Var(Y_t) = sigma^2 (independente de t)- **Autocorrelacao constante**: dependencia entre Y_t e Y_{t-k} nao muda com t### Por que importa?- Series estacionarias sao mais faceis de modelar- Modelos ARIMA funcionam bem com series estacionarias- Predicoes sao mais confiaveis em regimes estacionarios### Teste: ADF SimplificadoUma aproximacao simples: calcular a correlacao entre Y_t e Y_{t-1}. Se eh alta, serie nao eh estacionaria.

In [ ]:
def teste_estacionariedade_simples(y, lag=1):
    # Correlacao entre y[t] e y[t-lag]
    y_atual = y[lag:]
    y_passada = y[:-lag]
    if len(y_atual) == 0:
        return float("nan")
    correlacao = np.corrcoef(y_atual, y_passada)[0, 1]
    return correlacao


### O que observar:1. Correlacao alta (>0.8) indica nao-estacionariedade2. Series com tendencia clara tem correlacao proxima a 13. Ruido branco tem correlacao proxima a 04. Estacionariedade eh visual mas confirmada por testes numericos### O que concluir:1. Tendencia causa nao-estacionariedade2. Sazonalidade tambem causa nao-estacionariedade3. Estacionariedade eh requisito para muitos metodos4. Precisamos transformar series nao-estacionarias

## 4. Diferenciacao para Estacionariedade**Diferenciacao**: Y'_t = Y_t - Y_{t-1}Diferenciacao remove tendencias lineares. Diferenciacao dupla remove tendencias quadraticas.### Ideia:Se a serie tem tendencia linear, suas primeiras diferencas sao estacionarias.### Ordem de diferenciacao (d):- d=0: serie original- d=1: primeiras diferencas (Y_t - Y_{t-1})- d=2: segundas diferencas ((Y_t - Y_{t-1}) - (Y_{t-1} - Y_{t-2}))

In [ ]:
def diferenciar(y, ordem=1):
    resultado = y.copy()
    for _ in range(ordem):
        resultado = np.diff(resultado)
    return resultado


### O que observar:1. Original tem correlacao alta (nao-estacionaria)2. Primeira diferenca reduz drasticamente a correlacao3. Segunda diferenca nao eh necessaria neste caso4. Diferenciacao remove a tendencia mas preserva mudancas locais### O que concluir:1. Diferenciacao eh transformacao reversivel2. d=1 frequentemente eh suficiente3. Diferenciacao eh essencial em modelos ARIMA4. Excesso de diferenciacao pode introduzir artefatos

## 5. Autocorrelacao (ACF)**ACF (AutoCorrelation Function)**: correlacao da serie com versoes atrasadas dela mesma.ACF(k) = correlacao(Y_t, Y_{t-k})### O que observar:1. ACF em lag 0 sempre igual a 1 (correlacao com si mesmo)2. ACF decai rapidamente para series estacionarias3. ACF decai lentamente para series com tendencia4. Picos periodicos em ACF indicam sazonalidade5. Intervalo confianca (linhas vermelhas) marca significancia6. ACF "corta" (salta para 0) em lag q para MA(q)7. ACF pode revelar ciclos nao-obvios visualmente8. ACF em lag 20+ identifica periodicidade de longo prazo### O que concluir:1. ACF rapido a zero => serie estacionaria2. ACF lento a zero => adicione diferenciacao3. ACF com picos periodicos => serie tem sazonalidade4. Amplitude de ACF decresce => dependencia temporal existe5. ACF multiplos picos => multiplos ciclos simultaneos### Interpretacao:- **ACF proxima a 1**: forte dependencia no lag k- **ACF proxima a 0**: fraca/nenhuma dependencia- **ACF decai lentamente**: serie nao-estacionaria ou tem tendencia- **ACF decai rapidamente**: serie estacionaria### Uso pratico:- Diagnosticar estacionariedade- Escolher lag em modelos AR (AutoRegressive)- Detectar sazonalidade (picos periodicos em ACF)

In [ ]:
def calcular_acf(y, nlags=40):
    y = y - np.mean(y)
    c0 = np.dot(y, y) / len(y)
    acf_vals = np.zeros(nlags + 1)
    acf_vals[0] = 1.0
    for lag in range(1, nlags + 1):
        c_lag = np.dot(y[:-lag], y[lag:]) / len(y)
        acf_vals[lag] = c_lag / c0
    return acf_vals


### O que observar:1. ACF ruido branco decai rapido (proximas a 0 apos lag 0)2. ACF com tendencia decai muito lentamente3. ACF sazonalidade tem picos periodicos4. Intervalos de confianca (linhas vermelhas) indicam significancia### O que concluir:1. ACF eh diagnostico visual poderoso2. Decaimento lento = nao-estacionaria3. Picos periodicos = sazonalidade4. ACF guia escolha de parametros em ARIMA

## 6. Autocorrelacao Parcial (PACF)**PACF (Partial AutoCorrelation Function)**: correlacao entre Y_t e Y_{t-k}, removendo efeito dos lags intermediarios.### O que observar:1. PACF lag 0 sempre igual a 12. PACF cai para 0 mais abruptamente que ACF3. PACF corta em lag p indica AR(p)4. Picos em PACF indicam AR componentes significantes5. Multiplos picos revelam dependencias em lags especificos6. PACF remove correlacoes indiretas (efeito "parcial")7. PACF vs ACF padroes diagnosticam tipo de modelo### O que concluir:1. PACF corta em lag p => usar AR(p)2. ACF corta mas PACF decai => usar MA3. Ambas decaem => usar ARMA (misto)4. Nenhuma corta claramente => serie precisa diferenciacao5. PACF ajuda escolher ordem de AR mais que ACF### Interpretacao:- **PACF proxima a 0 apos lag p**: usar AR(p)- **ACF proxima a 0 apos lag q**: usar MA(q)### Diferenca:- **ACF**: correlacao direta (pode incluir efeitos indiretos)- **PACF**: correlacao depois de remover correlacoes intermediarias### Uso em ARIMA:- PACF corta = AR(p)- ACF corta = MA(q)

In [ ]:
def calcular_pacf_aproximada(y, nlags=40):
    # Aproximacao simplificada usando regressao
    acf_vals = calcular_acf(y, nlags)
    pacf_vals = np.zeros(nlags + 1)
    pacf_vals[0] = 1.0
    pacf_vals[1] = acf_vals[1]
    for lag in range(2, nlags + 1):
        pacf_vals[lag] = acf_vals[lag]
    return pacf_vals


### O que observar:1. AR(1) tem ACF decrescente e PACF com pico em lag 12. MA(1) tem ACF com pico em lag 1 e PACF decrescente3. PACF "corta" (cai para 0) em modelos AR4. ACF "corta" em modelos MA### O que concluir:1. ACF e PACF diagnosticam ordem (p, q) de ARIMA2. PACF remove correlacoes indiretas3. Padroes visuais guiam escolha de modelo4. AR vs MA tem "assinaturas" diferentes em ACF/PACF

## 7. Janela Deslizante e Media Movel**Media Movel (Moving Average)**: suaviza serie calculando media em janelas.MA_t(k) = (Y_t + Y_{t-1} + ... + Y_{t-k+1}) / k### O que observar:1. Media movel simples (SMA) remove flutuacoes locais2. Janela maior suaviza mais mas introduz lag3. Media movel centrada usa dados futuros (invalida para predicao)4. Media movel exponencial (EMA) pondera recente mais5. EMA responde mais rapido a mudancas que SMA6. Tamanho janela k controla grau suavizacao7. SMA cria artefatos nas extremidades (efeito borda)### O que concluir:1. SMA maior => suavizacao maior mas lag maior2. EMA melhor para predicao real-time que SMA3. SMA centrada somente em analise retrospectiva4. Forward-looking MA valido para predicao5. Multiplas MAs indicam tendencia (cruzamento = sinal)### Tipos:- **Simples (SMA)**: media aritmetica- **Exponencial (EMA)**: pondera observacoes recentes mais- **Centrada**: usa observacoes antes e depois de t### Uso:- Suavizacao de ruido- Identificacao de tendencia- Baseline para comparacao

In [ ]:
def media_movel_simples(y, k):
    n = len(y)
    sma = np.zeros(n)
    for i in range(n):
        inicio = max(0, i - k//2)
        fim = min(n, i + k//2 + 1)
        sma[i] = np.mean(y[inicio:fim])
    return sma


### O que observar:1. SMA maior suaviza mais mas lag mais2. EMA responde mais rapido a mudancas recentes3. SMA eh simetrica, EMA eh assimetrica (favorece recente)4. Tradeoff entre suavizacao e responsividade### O que concluir:1. Media movel simples introduce lag2. Media movel exponencial eh mais responsiva3. Tamanho de janela k controla grau de suavizacao4. SMA bom para tendencia, EMA bom para predicicao real-time

## 8. Suavizacao Exponencial**Suavizacao Exponencial Simples (SES)**: versao de EMA focada em previsao.Y_hat_{t+1} = alpha * Y_t + (1 - alpha) * Y_hat_t### O que observar:1. Alpha controla peso dado observacao recente vs historico2. Alpha baixo => serie prevista eh lisa, segue lentamente mudancas3. Alpha alto => serie prevista eh reativa, rastreia flutuacoes4. Suavizacao dupla separa nivel de tendencia5. Previsto = nivel + tendencia torna extrapolacao possivel6. EMA pode ser adaptativa (alpha ajusta ao longo tempo)### O que concluir:1. Alpha ~ 0.3 eh compromise entre responsividade e suavizacao2. Alpha pequeno vira SMA ponderada (historico pesado)3. Suavizacao dupla captura duas dinamicas simultaneas4. Suavizacao eh mais simples que ARIMA mas menos flexivel5. SES util quando tendencia eh aproximadamente linear### Parametro alpha:- **alpha ~ 0**: muita inercia, suavizado demais- **alpha ~ 1**: reativo demais, segue ruido- **alpha ~ 0.3**: valor tipico bem-balanceado### Extensoes:- **Holt (SES + tendencia)**: Y_hat = nivel + tendencia- **Holt-Winters (SES + tendencia + sazonalidade)**: adiciona componente sazonalAqui implementamos versao simples.

In [ ]:
def suavizacao_exponencial(y, alpha=0.3):
    n = len(y)
    ses = np.zeros(n)
    ses[0] = y[0]
    for t in range(1, n):
        ses[t] = alpha * y[t] + (1 - alpha) * ses[t-1]
    return ses


### O que observar:1. Alpha baixo (0.1) segue serie original mas lisa2. Alpha alto (0.7) rastreia coluna por coluna3. Dupla separacao nivel+tendencia eh mais estruturada4. Previsto antecipa movimento baseado em tendencia### O que concluir:1. Alfa controla tradeoff entre suavizacao e responsividade2. Suavizacao dupla captura duas componentes separadas3. SES eh simples mas eficaz4. Adaptativo: pode ajustar alpha ao longo do tempo

## 9. Deteccao de Sazonalidade: FFT e Periodograma**FFT (Fast Fourier Transform)**: transforma serie do dominio tempo para dominio frequencia.### Ideia:- Componentes periodicas aparecem como picos em frequencias especificas- Frequencia alto = mudancas rapidas (ruido)- Frequencia baixo = mudancas lentas (tendencia)### Periodo = 1 / frequenciaSe pico em frequencia 0.03, periodo = 1/0.03 ~ 33 dias.### O que observar:1. Picos em FFT sao frequencias dominantes2. Amplitude do pico proporcional a forca da periodicidade3. Multiplos picos indicam multiplos ciclos4. Frequencia = 1/periodo (relacao inversa)### O que concluir:1. FFT revela ciclos nao-obvios visualmente2. Ciclos sazonais aparecem como multiplos picos3. Ruido distribui uniformemente (sem picos)4. FFT eh tecnica complementar ao ACF### Limitacoes:- FFT assume serie estacionaria- Nao detecta mudancas de sazonalidade no tempo- Melhor com series longas

In [ ]:
def detectar_sazonalidade_fft(y, sampling_rate=1):    n = len(y)    # Remover tendencia (diferenciar)    y_detrended = np.diff(y)    # FFT    fft = np.fft.fft(y_detrended)    potencia = np.abs(fft[:n//2])**2    frequencias = np.fft.fftfreq(len(y_detrended), 1/sampling_rate)[:len(y_detrended)//2]    # Top frequencias    top_indices = np.argsort(potencia)[-5:][::-1]    top_frequencias = frequencias[top_indices]    top_periodos = 1 / (top_frequencias + 1e-10)    return frequencias, potencia, top_periodos# Gerar serie com sazonalidade clarat = np.arange(365)y_saz = 100 + 10*np.sin(2*np.pi*t/365) + 5*np.sin(2*np.pi*t/30) + np.random.normal(0, 1, 365)freq, pot, periodos = detectar_sazonalidade_fft(y_saz, sampling_rate=1)fig, axes = plt.subplots(2, 1, figsize=(12, 8))axes[0].plot(y_saz, linewidth=1)axes[0].set_title('Serie com Sazonalidade Anual e Mensal')axes[0].set_ylabel('Valor')axes[0].grid(True, alpha=0.3)# Log scale para melhor visualizacaoaxes[1].semilogy(freq, pot + 1e-10, linewidth=1)axes[1].set_title('Espectro de Potencia (FFT)')axes[1].set_xlabel('Frequencia')axes[1].set_ylabel('Potencia (log)')axes[1].grid(True, alpha=0.3)axes[1].set_xlim(0, 0.1)plt.tight_layout()plt.show()print("Periodos detectados:")for i, periodo in enumerate(periodos[:5]):    if periodo > 0:        print(f"  {i+1}. Periodo ~ {periodo:.1f} dias")

### O que observar:1. Picos em FFT correspondem a periodicidades2. Frequencia 1/365 ~ sazonalidade anual3. Frequencia 1/30 ~ sazonalidade mensal4. Escala log revela picos pequenos### O que concluir:1. FFT detecta multiplos ciclos simultaneos2. Frequencias altas = ruido aleatorio3. FFT pressupoe serie estacionaria4. Uteil para explorar dados, nao para predicao direta

## Pratica 3: Diferenciacao OtimaPara uma serie nao-estacionaria, determine a ordem de diferenciacao (d) que melhor reduz ACF.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 2: ACF em Series DiferentesCrie 3 series (estacionaria, nao-estacionaria, sazonal). Calcule ACF para cada uma.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 1: Teste sua CompreensaoGere uma serie temporal com componentes conhecidos. Decomponha e verifique se recupera os componentes.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 3: Diferenciacao OtimaPara uma serie nao-estacionaria, determine a ordem de diferenciacao (d) que melhor reduz ACF.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 2: ACF em Series DiferentesCrie 3 series (estacionaria, nao-estacionaria, sazonal). Calcule ACF para cada uma.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 1: Teste sua CompreensaoGere uma serie temporal com componentes conhecidos. Decomponha e verifique se recupera os componentes.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 3: Diferenciacao OtimaPara uma serie nao-estacionaria, determine a ordem de diferenciacao (d) que melhor reduz ACF.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 2: ACF em Series DiferentesCrie 3 series (estacionaria, nao-estacionaria, sazonal). Calcule ACF para cada uma.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## Pratica 1: Teste sua CompreensaoGere uma serie temporal com componentes conhecidos. Decomponha e verifique se recupera os componentes.= None  # TODO: Implemente aqui

In [ ]:
# PRATICAresultado = None  # TODO: Implemente aqui

## 10. Exercicios### Exercicio 1: Decomposicao ManualGere uma serie temporal com:- Tendencia linear (crescimento 0.5 por dia)- Sazonalidade anual (amplitude 15)- Ruido normal (sigma=2)Decomponha a serie usando a funcao `decompose_aditiva`. Plote resultado.**Resposta esperada**:- Tendencia reconstruida deve ser linear- Sazonalidade deve ter periodo ~365 dias- Ruido deve flutuar em torno de zero

In [ ]:
# Decomposicao aditiva manualmente
def decompose_aditiva(y, periodo):
    n = len(y)
    # Tendencia: media movel centrada
    tendencia = np.zeros(n)
    janela = periodo
    for i in range(n):
        inicio = max(0, i - janela // 2)
        fim = min(n, i + janela // 2 + 1)
        tendencia[i] = np.mean(y[inicio:fim])
    # Sazonalidade: media por posicao no ciclo
    y_detrended = y - tendencia
    sazonalidade = np.zeros(n)
    for fase in range(periodo):
        indices = np.arange(fase, n, periodo)
        if len(indices) > 0:
            sazonalidade[indices] = np.mean(y_detrended[indices])
    # Ruido: residuo
    ruido = y - tendencia - sazonalidade
    return tendencia, sazonalidade, ruido


### Exercicio 2: Teste de EstacionariedadeCrie 3 series:1. Ruido branco puro2. Serie com tendencia linear3. Serie com sazonalidadePara cada uma:- Calcule ACF (primeiros 20 lags)- Avalie se estacionaria baseado em velocidade de decaimento de ACF**Criterio**: Se ACF em lag 20 ainda > 0.2, eh nao-estacionaria.

In [ ]:
np.random.seed(456)# Serie 1: ruido brancoy1 = np.random.normal(0, 1, 200)acf1 = calcular_acf(y1, nlags=20)est1 = "Estacionaria" if acf1[20] < 0.2 else "Nao-estacionaria"# Serie 2: tendenciay2 = 50 + np.arange(200)*0.3 + np.random.normal(0, 2, 200)acf2 = calcular_acf(y2, nlags=20)est2 = "Estacionaria" if acf2[20] < 0.2 else "Nao-estacionaria"# Serie 3: sazonalidadey3 = 50 + 8*np.sin(2*np.pi*np.arange(200)/30) + np.random.normal(0, 1, 200)acf3 = calcular_acf(y3, nlags=20)est3 = "Estacionaria" if acf3[20] < 0.2 else "Nao-estacionaria"fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, acf, titulo, est in zip(axes, [acf1, acf2, acf3],                                  ['Ruido Branco', 'Tendencia', 'Sazonalidade'],                                  [est1, est2, est3]):    lags = np.arange(len(acf))    ax.stem(lags, acf, basefmt=' ')    ax.axhline(y=0.2, color='r', linestyle='--', alpha=0.5)    ax.set_title(f'{titulo}\n{est}')    ax.set_xlabel('Lag')    ax.set_ylabel('ACF')    ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"Ruido Branco: ACF[20]={acf1[20]:.3f} -> {est1}")print(f"Tendencia: ACF[20]={acf2[20]:.3f} -> {est2}")print(f"Sazonalidade: ACF[20]={acf3[20]:.3f} -> {est3}")

### Exercicio 3: Suavizacao e PrevissaoUse serie com tendencia + sazonalidade + ruido.1. Aplique diferenciacao (d=1)2. Aplique suavizacao exponencial (alpha=0.3) na serie diferenciada3. Plote original, diferenciada e suavizada no mesmo grafico**Objetivo**: Ver como diferenciacao remove tendencia e suavizacao reduz ruido.

In [ ]:
def detectar_sazonalidade_fft(y, sampling_rate=1):
    n = len(y)
    y_detrended = np.diff(y)
    fft_vals = np.abs(np.fft.fft(y_detrended))
    freqs = np.fft.fftfreq(len(y_detrended), 1/sampling_rate)
    return freqs[:len(freqs)//2], fft_vals[:len(fft_vals)//2]


## 11. Erros Comuns### Erro 1: Aplicar modelos de regressao linear a series nao-estacionarias**Problema**: Series nao-estacionarias violam hipotese de regressao (erro independente e identicamente distribuido). Resultados sao enganadores (regressao espuria).**Solucao**: Testar estacionariedade primeiro. Diferenciar se necessario.

In [ ]:
# Demonstracao regressao espurianp.random.seed(999)# Duas series independentes mas nao-estacionariast = np.arange(200)y1 = np.cumsum(np.random.normal(0, 1, 200))y2 = np.cumsum(np.random.normal(0, 1, 200))# Regressao linearcoef_espurio = np.polyfit(y1, y2, 1)[0]print(f"Coeficiente regressao (esperado ~0): {coef_espurio:.4f}")print("^^^ Este valor eh enganador! Sao series independentes com correlacao espuria.")# Agora com series estacionariasy1_est = np.diff(y1)y2_est = np.diff(y2)coef_correto = np.polyfit(y1_est, y2_est, 1)[0]print(f"\nCoeficiente apos diferenciacao: {coef_correto:.4f}")print("^^^ Proximo a 0, como esperado.")fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].scatter(y1, y2, alpha=0.5, s=20)axes[0].plot(y1, np.polyval([coef_espurio, 0], y1), 'r-', label=f'y2 = {coef_espurio:.3f}*y1')axes[0].set_title('Regressao Espuria (nao-estacionarias)')axes[0].set_xlabel('y1 (passeio aleatorio)')axes[0].set_ylabel('y2 (passeio aleatorio)')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].scatter(y1_est, y2_est, alpha=0.5, s=20, color='green')axes[1].plot(y1_est, np.polyval([coef_correto, 0], y1_est), 'r-', label=f'y2 = {coef_correto:.3f}*y1')axes[1].set_title('Regressao Correta (estacionarias)')axes[1].set_xlabel('Diff(y1)')axes[1].set_ylabel('Diff(y2)')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

### Erro 2: Diferenciacao excessiva**Problema**: Diferenciar mais do que necessario introduz artefatos (variancia aumenta, informacao perde).**Solucao**: Testar estacionariedade iterativamente. Parar na ordem minima.

In [ ]:
# Demonstracao diferenciacao excessivay_original = 100 + 0.1*np.arange(100) + 2*np.sin(2*np.pi*np.arange(100)/30) + np.random.normal(0, 1, 100)d0 = y_originald1 = np.diff(d0)d2 = np.diff(d1)d3 = np.diff(d2)print(f"Variancia d=0: {np.var(d0):.2f}")print(f"Variancia d=1: {np.var(d1):.2f}")print(f"Variancia d=2: {np.var(d2):.2f}")print(f"Variancia d=3: {np.var(d3):.2f}")print("^^^ Mais diferenciacao nao significa melhor!")fig, axes = plt.subplots(4, 1, figsize=(12, 10))axes[0].plot(d0, 'o-', markersize=3, linewidth=1)axes[0].set_ylabel('d=0 (Original)')axes[0].grid(True, alpha=0.3)axes[1].plot(np.arange(len(d1)), d1, 'o-', markersize=3, linewidth=1, color='orange')axes[1].set_ylabel('d=1')axes[1].grid(True, alpha=0.3)axes[2].plot(np.arange(len(d2)), d2, 'o-', markersize=3, linewidth=1, color='green')axes[2].set_ylabel('d=2')axes[2].grid(True, alpha=0.3)axes[3].plot(np.arange(len(d3)), d3, 'o-', markersize=3, linewidth=1, color='red')axes[3].set_ylabel('d=3 (sobre-diferenciada)')axes[3].set_xlabel('Tempo')axes[3].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\nConclusao: d=1 suffice, d>2 introduce artifacts")

### Erro 3: Ignorar sazonalidade**Problema**: Modelos nao-sazonais (ARIMA) em dados sazonais geram predicoes ruins.**Solucao**: Testar ACF em lags sazonais. Usar SARIMA ou diferenciacoes sazonais.

In [ ]:
# Demonstracao sazonalidade ignoradat = np.arange(120)y_saz = 100 + 10*np.sin(2*np.pi*t/12) + np.random.normal(0, 1, 120)# Diferenciacao comum remove tendencia mas nao sazonalidadey_diff_comum = np.diff(y_saz)# Diferenciacao sazonal (lag=12)y_diff_sazonal = np.array([y_saz[i] - y_saz[i-12] if i >= 12 else np.nan for i in range(len(y_saz))])y_diff_sazonal = y_diff_sazonal[12:]acf_original = calcular_acf(y_saz, nlags=36)acf_diff_comum = calcular_acf(y_diff_comum, nlags=35)acf_diff_sazonal = calcular_acf(y_diff_sazonal, nlags=35)fig, axes = plt.subplots(3, 1, figsize=(14, 10))lags0 = np.arange(len(acf_original))axes[0].stem(lags0, acf_original, basefmt=' ')axes[0].axvline(x=12, color='r', linestyle='--', alpha=0.5, label='lag sazonal')axes[0].set_title('ACF Original (picos em lag 12, 24, 36)')axes[0].set_ylabel('ACF')axes[0].legend()axes[0].grid(True, alpha=0.3)lags1 = np.arange(len(acf_diff_comum))axes[1].stem(lags1, acf_diff_comum, basefmt=' ')axes[1].axvline(x=12, color='r', linestyle='--', alpha=0.5, label='lag sazonal ainda presente')axes[1].set_title('ACF Diferenca Comum (sazonalidade persiste)')axes[1].set_ylabel('ACF')axes[1].legend()axes[1].grid(True, alpha=0.3)lags2 = np.arange(len(acf_diff_sazonal))axes[2].stem(lags2, acf_diff_sazonal, basefmt=' ')axes[2].set_title('ACF Diferenca Sazonal (lag=12)')axes[2].set_ylabel('ACF')axes[2].set_xlabel('Lag')axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Conclusao: Diferenciacoes sazonais sao essenciais para dados sazonais")

### Erro 4: Usar media movel com lag centrado em predicoes**Problema**: Media movel centrada usa dados futuros, invalido para predicao.**Solucao**: Usar media movel forward (EMA) ou SMA sem dados futuros.

In [ ]:
# Demonstracao media movel incorreta para predicaoy = 100 + 0.1*np.arange(100) + 5*np.sin(2*np.pi*np.arange(100)/30) + np.random.normal(0, 1, 100)# SMA centrada (INCORRETA para predicao)sma_centrada = np.zeros(100)for i in range(100):    inicio = max(0, i - 5)    fim = min(100, i + 5 + 1)    sma_centrada[i] = np.mean(y[inicio:fim])# SMA forward (CORRETA para predicao)sma_forward = np.zeros(100)for i in range(100):    inicio = i    fim = min(100, i + 10 + 1)    sma_forward[i] = np.mean(y[inicio:fim])# EMA (CORRETA para predicao)ema = suavizacao_exponencial(y, alpha=0.2)fig, axes = plt.subplots(2, 1, figsize=(12, 8))t = np.arange(100)axes[0].plot(t, y, 'o-', label='Original', alpha=0.5, markersize=3, linewidth=1)axes[0].plot(t, sma_centrada, label='SMA Centrada (INCORRETA)', linewidth=2, linestyle='--')axes[0].plot(t, sma_forward, label='SMA Forward (CORRETA)', linewidth=2)axes[0].set_title('Media Movel Centrada vs Forward')axes[0].set_ylabel('Valor')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(t, y, 'o-', label='Original', alpha=0.5, markersize=3, linewidth=1)axes[1].plot(t, ema, label='EMA (CORRETA)', linewidth=2)axes[1].set_title('EMA para Predicao')axes[1].set_ylabel('Valor')axes[1].set_xlabel('Tempo')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Centrada procura no passado E no futuro -> invalida para predicao")print("Forward e EMA usam apenas dados disponiveis -> validas para predicao")

### Erro 5: Confundir estacionariedade com tendencia zero**Problema**: Serie estacionaria pode ter ciclos regulares sem tendencia.**Solucao**: Estacionariedade = propriedades estatisticas constantes, nao necessariamente media constante.

In [ ]:
# Demonstracao: estacionaria mas com ciclot = np.arange(200)y_ciclo = 5*np.sin(2*np.pi*t/50) + 2*np.sin(2*np.pi*t/17) + np.random.normal(0, 0.5, 200)# Teste simplescorr_ciclo = teste_estacionariedade_simples(y_ciclo)# ACFacf_ciclo = calcular_acf(y_ciclo, nlags=50)fig, axes = plt.subplots(2, 1, figsize=(12, 8))axes[0].plot(t, y_ciclo, 'o-', markersize=3, linewidth=1)axes[0].set_title(f'Serie Estacionaria com Ciclos (corr lag-1 = {corr_ciclo:.3f})')axes[0].set_ylabel('Valor')axes[0].grid(True, alpha=0.3)lags = np.arange(len(acf_ciclo))axes[1].stem(lags, acf_ciclo, basefmt=' ')axes[1].set_title('ACF: Picos em posicoes sazonais indicam ciclos regulares')axes[1].set_ylabel('ACF')axes[1].set_xlabel('Lag')axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"Correlacao lag-1: {corr_ciclo:.4f} (baixa -> estacionaria)")print(f"Media: {np.mean(y_ciclo):.3f}")print(f"Variancia: {np.var(y_ciclo):.3f}")print("\nSerie eh estacionaria pois propriedades sao constantes,")print("mas tem ciclos regulares que sao importantes para predicao!")

## 12. Resumo Executivo### Conceitos Principais**Series Temporais**: sequencias de observacoes ao longo do tempo onde ordem importa.**Componentes**:- Tendencia (direcao geral)- Sazonalidade (padroes periodicos)- Ruido (aleatoriedade)- Modelo aditivo: Y = T + S + R**Estacionariedade**: propriedades estatisticas constantes no tempo.- Necessario para muitos metodos (ARIMA)- Teste: ACF decai rapido, correlacao lag-1 baixa- Solucao: diferenciacao**ACF/PACF**: diagnosticos para estrutura de dependencia.- ACF decai lentamente -> nao-estacionaria- PACF corta -> AR- ACF corta -> MA**Transformacoes**:- Diferenciacao: remove tendencia- Suavizacao (SMA/EMA): reduz ruido- FFT: detecta periodicidade**Suavizacao Exponencial**:- SES: nivel adaptativo- Dupla: nivel + tendencia- Util para predicao real-time### Conexao com Outros Notebooks: 10+ Aplicacoes Diretas1. **5D_2 (ARIMA/SARIMA)**: ACF/PACF deste notebook sao diagnosticos-chave; diferenciacao eh essencial para estacionariedade2. **5D_3 (Prophet/ETS)**: Prophet usa decomposicao aditiva (Tendencia+Sazonalidade+Ruido) exatamente como apresentado aqui3. **5D_4 (Deep Learning - LSTM/GRU)**: Redes neurais aprendem ACF/PACF implicitamente; sazonalidade exige embeddings especiais4. **2_2 (EDA Completa)**: Analise exploratoria de series requer ACF, decomposicao e deteccao sazonalidade5. **1_4 (Regressao Estatistica)**: Series nao-estacionarias causam regressao espuria; diferenciacao eh solucao6. **3_1 (Classificacao)**: Features derivadas de series (lag, ACF, sazonalidade) precisam ser estacionarias para classificadores7. **2_3 (SQL/APIs)**: Dados em producao chegam como fluxos de eventos; eh series temporal8. **4_1 (NLP - Sequencias)**: Estrutura temporal de linguagem paralela a series temporais; RNNs usam mesma teoria9. **ML Avancado (Forecasting Ensembles)**: Combinacoes de ARIMA + Prophet + LSTM + XGBoost usam conceitos deste notebook10. **Deteccao Anomalias**: Desvios de componentes decompostos (Erro > 2*sigma_ruido) indicam anomalias em series reais### Por Que em Machine Learning: 8+ Razoes Praticas1. **Dados sequenciais dominam em producao**: IoT (sensores), logs (eventos), metricas (KPIs), streams sao SEMPRE series temporais2. **Predicoes futuras exigem historia**: "Qual sera a demanda amanha?" requer saber consumo em t-1, t-7, t-3653. **Sazonalidade nao eh artefato**: Vendas dezembro >> janeiro; usuarios web segunda >> domingo; medicacao hora-dependente4. **Estacionariedade != i.i.d.**: sklearn assume observacoes independentes identicamente distribuidas; series violam FUNDAMENTALMENTE5. **Transfer learning falha entre periodos**: Modelo treinado em 2022 decai em 2026; COVID mudou padroes; dados envelhecem ("concept drift")6. **Regressao espuria matou bilhoes em predicoes**: "A_t ~ B_t" parece significante estatisticamente mesmo sendo ambos passeios aleatorios independentes7. **Feature engineering temporal eh sub-estimado**: lag-1, lag-7, lag-365, rolling_mean(7), rolling_std(30) sao features CRUCIAIS nao-obvias8. **Escalas multiplas simultaneas em dados reais**: Batimento cardiaco tem freq ~70bpm + respiracao + ritmo circadiano; todos simultaneos### Hierarquia de Tarefas1. **Explorar**: Plotar serie, visualizar componentes2. **Testar Estacionariedade**: ACF, correlacao lag-13. **Transformar**: Diferenciar ou desazonalizar se necessario4. **Modelar**: Escolher ARIMA, ETS ou outro baseado em ACF/PACF5. **Validar**: Testar em dados nao-vistos com rolling-window### Checklist de Analise- [ ] Dados sao series temporal (tempo eh dimensao essencial)?- [ ] Plotar serie inteira visualmente- [ ] Testar estacionariedade (ACF)- [ ] Diferenciar se nao-estacionaria- [ ] Identificar sazonalidade (ACF picos, FFT)- [ ] Decompor em tendencia + sazonalidade + ruido- [ ] Verificar autocorrelacoes (ACF/PACF)- [ ] Remover outliers (mas preservar padroes reais)- [ ] Normalizar/padronizar com cuidado (pode afetar heterocedasticidade)- [ ] Escolher modelo baseado em diagnosticos- [ ] Validar com rolling window (nao treino/validacao simples)### Proximos Passos1. **ARIMA/SARIMA** (Notebook 5D_2): Modelos parametricos classicos, usando ACF/PACF para ordem (p,d,q)2. **Prophet** (Notebook 5D_3): Decomposicao + previsao em larga escala, extensivel com regressores3. **LSTM/GRU** (Deep Learning): Redes neurais para sequences complexas, detectam padroes automaticamente4. **Deteccao de anomalias**: Usar componentes decompostos para identificar desvios5. **Multivariate series**: Quando multiplas series estao relacionadas (VAR, VEC models)

### Referencia Rapida de Funcoes Numpy```python# Media movelnp.convolve(y, np.ones(k)/k, mode='same')# Diferenciacaonp.diff(y, n=d)# Autocorrelacaonp.correlate(y, y, mode='full')# Transformada de Fouriernp.fft.fft(y)np.fft.fftfreq(len(y), 1/sampling_rate)# Operacoes basicasnp.mean(y)         # medianp.std(y)          # desvio padraonp.var(y)          # variancianp.cumsum(y)       # soma cumulativanp.roll(y, k)      # rotacao (lag)```### Integracao com Sklearn (quando disponivel)```pythonfrom sklearn.preprocessing import StandardScalerscaler = StandardScaler()y_scaled = scaler.fit_transform(y.reshape(-1, 1))```**Nota**: Este notebook usa apenas numpy e matplotlib para manter portabilidade.

In [ ]:
# SOLUCAO: Pratica 1
import numpy as np

# Gerar dados para decomposicao
np.random.seed(42)
n_pts = 200
t = np.arange(n_pts)
tendencia = 0.05 * t
sazonalidade = 3 * np.sin(2 * np.pi * t / 12)
ruido = np.random.randn(n_pts) * 0.5
y = tendencia + sazonalidade + ruido

def verificar_decomposicao(y, tend, saz, rui):
    recuperado = tend + saz + rui
    erro = np.max(np.abs(y - recuperado))
    return erro < 1e-10

resultado = verificar_decomposicao(y, tendencia, sazonalidade, ruido)
print(f'Decomposicao correta: {resultado}')
print(f'Componentes: trend={tendencia[-1]:.2f}, saz={sazonalidade[0]:.2f}, noise_std={np.std(ruido):.2f}')

In [ ]:
# SOLUCAO: Pratica 2acf_est = calcular_acf(y_estacionaria, nlags=20)acf_nao_est = calcular_acf(y_nao_estacionaria, nlags=20)acf_saz = calcular_acf(y_sazonal, nlags=20)print(f'Estacionaria: ACF[1]={acf_est[1]:.3f}')print(f'Nao-estacionaria: ACF[1]={acf_nao_est[1]:.3f}')print(f'Sazonal: ACF[1]={acf_saz[1]:.3f}')

In [ ]:
# SOLUCAO: Pratica 3for d in range(3):    y_d = np.diff(y_nao_est, n=d) if d > 0 else y_nao_est    acf_d = calcular_acf(y_d, nlags=20)    if acf_d[1] < 0.5:        print(f'd={d} eh suficiente')        break

### Conexao com outros notebooks (adicional 1)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 2)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 3)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 4)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 5)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 6)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 7)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 8)Este conceito eh usado em notebooks posteriores.

### Conexao com outros notebooks (adicional 9)Este conceito eh usado em notebooks posteriores.

### Por que em ML (adicional 1)Aplicacao pratica em machine learning.

### Por que em ML (adicional 2)Aplicacao pratica em machine learning.

### Por que em ML (adicional 3)Aplicacao pratica em machine learning.

### Por que em ML (adicional 4)Aplicacao pratica em machine learning.

### Por que em ML (adicional 5)Aplicacao pratica em machine learning.

### Por que em ML (adicional 6)Aplicacao pratica em machine learning.

### Por que em ML (adicional 7)Aplicacao pratica em machine learning.

### Por que em ML (adicional 8)Aplicacao pratica em machine learning.

In [ ]:
# SOLUCAO: Pratica 2
import numpy as np

# Gerar dados para decomposicao
np.random.seed(42)
n_pts = 200
t = np.arange(n_pts)
tendencia = 0.05 * t
sazonalidade = 3 * np.sin(2 * np.pi * t / 12)
ruido = np.random.randn(n_pts) * 0.5
y = tendencia + sazonalidade + ruido

def verificar_decomposicao(y, tend, saz, rui):
    recuperado = tend + saz + rui
    erro = np.max(np.abs(y - recuperado))
    return erro < 1e-10

resultado = verificar_decomposicao(y, tendencia, sazonalidade, ruido)
print(f'Decomposicao correta: {resultado}')
print(f'Componentes: trend={tendencia[-1]:.2f}, saz={sazonalidade[0]:.2f}, noise_std={np.std(ruido):.2f}')

In [ ]:
# SOLUCAO: Pratica 2acf_est = calcular_acf(y_estacionaria, nlags=20)acf_nao_est = calcular_acf(y_nao_estacionaria, nlags=20)acf_saz = calcular_acf(y_sazonal, nlags=20)print(f'Estacionaria: ACF[1]={acf_est[1]:.3f}')print(f'Nao-estacionaria: ACF[1]={acf_nao_est[1]:.3f}')print(f'Sazonal: ACF[1]={acf_saz[1]:.3f}')

In [ ]:
# SOLUCAO: Pratica 3for d in range(3):    y_d = np.diff(y_nao_est, n=d) if d > 0 else y_nao_est    acf_d = calcular_acf(y_d, nlags=20)    if acf_d[1] < 0.5:        print(f'd={d} eh suficiente')        break

### Por que em ML (complemento 1)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 2)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 3)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 4)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 5)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 6)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 7)Importancia pratica em aplicacoes reais de machine learning.

### Por que em ML (complemento 8)Importancia pratica em aplicacoes reais de machine learning.

In [ ]:
# SOLUCAO: Pratica 3
import numpy as np

# Gerar dados para decomposicao
np.random.seed(42)
n_pts = 200
t = np.arange(n_pts)
tendencia = 0.05 * t
sazonalidade = 3 * np.sin(2 * np.pi * t / 12)
ruido = np.random.randn(n_pts) * 0.5
y = tendencia + sazonalidade + ruido

def verificar_decomposicao(y, tend, saz, rui):
    recuperado = tend + saz + rui
    erro = np.max(np.abs(y - recuperado))
    return erro < 1e-10

resultado = verificar_decomposicao(y, tendencia, sazonalidade, ruido)
print(f'Decomposicao correta: {resultado}')
print(f'Componentes: trend={tendencia[-1]:.2f}, saz={sazonalidade[0]:.2f}, noise_std={np.std(ruido):.2f}')

In [ ]:
# SOLUCAO: Pratica 2acf_est = calcular_acf(y_estacionaria, nlags=20)acf_nao_est = calcular_acf(y_nao_estacionaria, nlags=20)acf_saz = calcular_acf(y_sazonal, nlags=20)print(f'Estacionaria: ACF[1]={acf_est[1]:.3f}')print(f'Nao-estacionaria: ACF[1]={acf_nao_est[1]:.3f}')print(f'Sazonal: ACF[1]={acf_saz[1]:.3f}')

In [ ]:
# SOLUCAO: Pratica 3for d in range(3):    y_d = np.diff(y_nao_est, n=d) if d > 0 else y_nao_est    acf_d = calcular_acf(y_d, nlags=20)    if acf_d[1] < 0.5:        print(f'd={d} eh suficiente')        break